In [ ]:
import torch
from rdkit import Chem, RDLogger
from src.mlconfgen import MLConformerGenerator, evaluate_samples
from src.mlconfgen.utils import standardize_mol
from pharmacophore import color_tanimoto

RDLogger.DisableLog('rdApp.*')

if torch.cuda.is_available():
    device = torch.device("cuda:0")
elif torch.backends.mps.is_available():
    device = torch.device("mps:0")
else:
    device = torch.device("cpu")

print(f"Intitialising model on {device}")

generator = MLConformerGenerator(
                                 edm_weights="./edm_moi_chembl_15_39.pt",
                                 adj_mat_seer_weights="./adj_mat_seer_chembl_15_39.pt",
                                 device=device,
                                 diffusion_steps=10,
                                )

ref_mol = Chem.MolFromMolFile('./assets/demo_files/yibfeu.mol')

def score(mol):
    try:
        mol = standardize_mol(mol, optimize_geometry=True)
        ref_mb, scores = evaluate_samples(ref_mol, [mol])
        aligned_ref = Chem.MolFromMolBlock(ref_mb)
        aligned_cand = Chem.MolFromMolBlock(scores[0]['mol_block'])

        color_sim = color_tanimoto(aligned_ref, aligned_cand)
        return color_sim
    except:
        return 0
    

generator.fine_tune(
                  score_function=score,  # This should output normalised score from (-1, 1)
                  reference_conformer=ref_mol,
                  variance= 1,
                  # RL Fine-tune params
                  n_epochs=100,
                  batch_size=16,
                  learning_rate= 8e-5,
                  sigma=60.0,
                  temperature=1.5,
                  n_samples_per_mol=8,
                  reward_clip=(-1.0, 1.0),
                  eval_every=1,
                  save_dir="./rl_checkpoints_validity",
    
)



[10:52:05] Initializing Normalizer


Intitialising model on mps:0
[Epoch 0001/0100] loss=990.0226 reward_mean=-0.1284 valid_rate=0.7500 agent_ll=-2.9181 prior_ll=-2.9181
                 agent_score_mean=-0.1731 baseline_score_mean=-0.3827
                 eval_agent_valid_rate=0.7188 eval_baseline_valid_rate=0.5312
                 score_improv=0.2095 valid_rate_improv=0.1875 
                 saved new best agent head
[Epoch 0002/0100] loss=1583.0244 reward_mean=-0.3645 valid_rate=0.5703 agent_ll=-3.2842 prior_ll=-3.3437
                 agent_score_mean=-0.2005 baseline_score_mean=-0.4312
                 eval_agent_valid_rate=0.6875 eval_baseline_valid_rate=0.4688
                 score_improv=0.2307 valid_rate_improv=0.2188 
                 saved new best agent head


In [ ]:
import time
import torch

from rdkit import Chem, RDLogger
from rdkit.Chem import Draw

from mlconfgen import MLConformerGenerator, evaluate_samples



generator = MLConformerGenerator(
                                 edm_weights="./edm_moi_chembl_15_39.pt",
                                 adj_mat_seer_weights="./adj_mat_seer_chembl_15_39.pt",
                                 device=device,
                                 diffusion_steps=20,
                                )

# generator.adj_mat_seer.resize.load_state_dict(torch.load("./rl_checkpoints_long_run_2/best_agent_resize.pt"))

# Load a Reference conformer
ref_mol = Chem.MolFromMolFile('./assets/demo_files/yibfeu.mol')


# Generate Samples
print("Generation started...")
start = time.time()

# Resampling significantly increases generation quality, while sacrificing speed
N_SAMPLES = 20
samples = generator.generate_conformers(
                                        reference_conformer=ref_mol,
                                        n_samples=N_SAMPLES,
                                        variance=1,
                                        resample_steps=0,
                                        )

print(f"Generation complete in {round(time.time() - start, 2)}")

# Characterise samples   
_, std_samples = evaluate_samples(ref_mol, samples)

# Display results
mols = []
legends = []
average_shape_similarity = 0
for sample in std_samples:
    mol = Chem.MolFromMolBlock(sample['mol_block'])
    mol = Chem.MolFromSmiles(Chem.MolToSmiles(mol))
    mol.SetProp("Shape_Tanimoto", str(sample['shape_tanimoto']))
    mols.append(mol)
    legends.append(f"Shape Similarity - {round(sample['shape_tanimoto'], 2)}")
    average_shape_similarity += round(sample['shape_tanimoto'], 2)

average_shape_similarity = average_shape_similarity / len(std_samples)
print(f"AVERAGE SHAPE SIMILARITY - {average_shape_similarity}")
print(f"VALID SAMPLES - {round(len(std_samples) / N_SAMPLES, 2) * 100} %")
    
Draw.MolsToGridImage(mols, legends=legends)